In [8]:
import os
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

# 데이터 확인하기 2025.11.21 zero_count_rate > 99% 이상인 컬럼 제거 후 RandomForest + 상관계수 높은 컬럼 삭제 후 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.ensemble      import RandomForestClassifier 

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
import utils.preprocessing as preprocessing
import utils.user_utils    as user_utils


# 모듈 reload
importlib.reload(preprocessing)
importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns, remove_zero_columns2
from utils.user_utils    import get_model_train_eval



In [ ]:
# 데이터 로딩
train, test = load_data()

# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거 

# Data 전처리 1. zero_count_rate이 95%인 컬럼 제거하기 
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.99)

# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

# 스케일링
X_train_scaled, X_test_scaled, scaler = scale_data(X_features, X_test)

# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features, 
  y_labels,
)



In [9]:
model_name = 'RandomForest_99per_HP_ne390_maxDepth25_classWeight12_msl1_mss7' 

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 390,
  max_depth    = 25, 
  class_weight = {0:1, 1:2}, # 클래스별 가중치
  min_samples_leaf = 1, 
  min_samples_split = 7,
  n_jobs       = -1 # 병렬처리 여부 
)

# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)

✓ 모델 저장 완료: ../models\RandomForest_99per_HP_ne390_maxDepth25_classWeight12_msl1_mss7.pkl
  파일 크기: 77.91 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8374, 정확도: 0.9603, 정밀도: 0.4545, 재현율: 0.0083, F1: 0.0163
오차행렬:
[[14596     6]
 [  597     5]]
실행 시간: 12.398780345916748


In [ ]:
# first testing model
# XGBoost (xgb) : yjh, kjh
# LightGBM(lgbm) : lsj, ujm
# Random Forest(rf) : lkj, kjh
# Logistic Regression(lr) : yjh, ujm


In [10]:
# # 양성(1)과 음성(0) 데이터 분리
# pos = X_features[train.TARGET == 1]
# neg = X_features[train.TARGET == 0]

# print("양성 데이터 수:", len(pos))
# print("음성 데이터 수:", len(neg))

# # 원하는 비율 설정 (예: 10배)
# ratio = 10

# # 음성 데이터에서 샘플링
# neg_sampled = neg.sample(n=len(pos) * ratio, random_state=42)

# # 결합하여 새로운 학습 데이터 생성
# balanced = pd.concat([pos, neg_sampled])

# print("최종 데이터셋 크기:", balanced.shape)
# print("양성:음성 비율 =", len(pos), ":", len(neg_sampled))

양성 데이터 수: 3008
음성 데이터 수: 73012
최종 데이터셋 크기: (33088, 149)
양성:음성 비율 = 3008 : 30080


In [17]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features, 
  y_labels,
)


In [18]:
model_name = 'RandomForest_99per_HP_ne390_maxDepth25_classWeight12_msl1_mss7' 

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 390,
  max_depth    = 25, 
  class_weight = {0:1, 1:2}, # 클래스별 가중치
  min_samples_leaf = 1, 
  min_samples_split = 7,
  n_jobs       = -1 # 병렬처리 여부 
)

# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)
# model_name = 'RandomForest_99per_HP_ne300_maxDepth20_classWeight'
# AUC: 0.8412, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0050, F1: 0.0099 

# AUC: 0.8374, 정확도: 0.9603, 정밀도: 0.4545, 재현율: 0.0083, F1: 0.0163  # Best ★★★

✓ 모델 저장 완료: models\RandomForest_99per_HP_ne390_maxDepth25_classWeight12_msl1_mss7.pkl
  파일 크기: 77.91 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8374, 정확도: 0.9603, 정밀도: 0.4545, 재현율: 0.0083, F1: 0.0163
오차행렬:
[[14596     6]
 [  597     5]]
실행 시간: 11.974271297454834
